# CSV에서 channel_id 추출 (YT_ChannelData — 5개 카테고리)

vling에서 다운로드한 `*_raw.csv` 파일들의 `Channel Link` 컬럼에서  
YouTube `channel_id`를 추출해 새 컬럼으로 추가하고, 중복 채널을 제거합니다.

## 처리 대상 파일

| 파일 | 카테고리 |
|---|---|
| `YT_ChannelData_Health_raw.csv` | 헬스/운동 |
| `YT_ChannelData_food_raw.csv` | 음식/요리 |
| `YT_ChannelData_VLOG_raw.csv` | 브이로그 |
| `YT_ChannelData_society_raw.csv` | 사회/시사 |
| `YT_ChannelData_education_raw.csv` | 교육 |

## Channel Link → channel_id 추출 규칙

```
https://vling.net/channel/UChXDq9Izwq-pTzeUWzwXL3A
                          ↑ UC 로 시작하는 이 부분이 channel_id
```

각 카테고리별로 `channel_id` 중복 행을 제거(첫 번째 유지)하고  
`*_clean.csv` 파일로 저장합니다.

In [7]:
import re
import pandas as pd
from pathlib import Path

# 처리할 카테고리 목록 (raw 파일명 접두사)
CATEGORIES = ["Health", "food", "VLOG", "society", "education"]

# 파일 존재 여부 확인
for cat in CATEGORIES:
    raw = Path(f"YT_ChannelData_{cat}_raw.csv")
    status = "✓" if raw.exists() else "✗ 없음"
    print(f"  [{status}] {raw.name}")

print("\n처리 준비 완료").csv")

df_raw = pd.read_csv(CHANNEL_DATA_CSV, encoding="utf-8-sig")

print(f"로드 완료: {len(df_raw)}개 채널")
print(f"컬럼: {df_raw.columns.tolist()}")
df_raw.head(5)

로드 완료: 1000개 채널
컬럼: ['Channel Name', 'Channel Link', 'Subscribers', 'Average Daily Views']


,Channel Name,Channel Link,Subscribers,Average Daily Views
0,다이어트 과학자 최겸 Gyum Choi,https://vling.net/channel/UChXDq9Izwq-pTzeUWzw...,603000,79982
1,한의사트레이너,https://vling.net/channel/UC_KjTyC5e0DFNxn6GIg...,144000,35569
2,블락스blocks,https://vling.net/channel/UC5qF47U0ErUHBgK4Rgy...,10100,294
3,김지만TV,https://vling.net/channel/UCCFD5pLnLB2oEb4tPm1...,141000,995
4,김소영건강체조티비-Health gymnastics,https://vling.net/channel/UCSi2gkWNFwoMpb5Nu19...,53000,16501


### channel_id 추출 및 중복 제거

`Channel Link` 컬럼 URL에서 `UC`로 시작하는 채널 ID를 파싱하고,  
같은 `channel_id`가 여러 번 등장하면 첫 번째 행만 남깁니다.

In [8]:
_ID_RE = re.compile(r"/(UC[\w-]+)")

def process_raw(category: str) -> pd.DataFrame:
    """raw CSV 로드 → channel_id 추출 → 중복 제거 → clean CSV 저장."""
    raw_path   = Path(f"YT_ChannelData_{category}_raw.csv")
    clean_path = Path(f"YT_ChannelData_{category}_clean.csv")

    df = pd.read_csv(raw_path, encoding="utf-8-sig")

    # channel_id 추출
    df["channel_id"] = (
        df["Channel Link"]
        .fillna("")
        .apply(lambda url: m.group(1) if (m := _ID_RE.search(url)) else "")
    )

    # channel_id 컬럼을 Channel Link 바로 뒤에 삽입 (원본 컬럼 유지)
    link_pos = df.columns.get_loc("Channel Link") + 1
    df.insert(link_pos, "channel_id", df.pop("channel_id"))

    # channel_id 없는 행 제거
    before_drop = len(df)
    df = df[df["channel_id"] != ""].reset_index(drop=True)
    no_id = before_drop - len(df)

    # 중복 channel_id 제거 (첫 번째 행 유지)
    before_dedup = len(df)
    df = df.drop_duplicates(subset="channel_id", keep="first").reset_index(drop=True)
    duplicates = before_dedup - len(df)

    df.to_csv(clean_path, index=False, encoding="utf-8-sig")

    print(f"[{category:12s}]  원본: {before_drop:4d}  "
          f"ID 없음: {no_id:3d}  중복 제거: {duplicates:3d}  "
          f"최종: {len(df):4d}  → {clean_path.name}")
    return df


results = {}
for cat in CATEGORIES:
    results[cat] = process_raw(cat)

print("\n처리 완료!")

channel_id 추출 성공: 1000 / 1000


,channel_name,channel_id,subscribers,avg_daily_views
0,다이어트 과학자 최겸 Gyum Choi,UChXDq9Izwq-pTzeUWzwXL3A,603000,79982
1,한의사트레이너,UC_KjTyC5e0DFNxn6GIgBxcQ,144000,35569
2,블락스blocks,UC5qF47U0ErUHBgK4Rgy5ymA,10100,294
3,김지만TV,UCCFD5pLnLB2oEb4tPm1YemA,141000,995
4,김소영건강체조티비-Health gymnastics,UCSi2gkWNFwoMpb5Nu19EnpA,53000,16501
5,통증교정 수연쌤,UCafAiwzaq1wXHIJOCm4zSoQ,12400,1055
6,우수한 영양학,UCxkwgIYCm3vLlYITOzdByZg,62800,7539
7,빵느,UCRrZ5RYIalHLiHq5ftzxM6A,711000,66720
8,홍삼아 댄스&에어로빅,UCNcl4-NKaNrX0CBK3Gmd5Ag,21400,3136
9,약초1번지,UCvf59uLGe-VyEcoNsp15K0w,102000,532


### 결과 미리보기

각 카테고리별 clean CSV의 상위 3개 행을 확인합니다.

In [9]:
OUTPUT_CSV = Path("YT_ChannelData_2026-03-28_clean.csv")

df_result = df_clean[df_clean["channel_id"] != ""].reset_index(drop=True)
df_result.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"저장 완료: {len(df_result)}개 채널 → {OUTPUT_CSV}")
print(f"channel_id 없어서 제외: {len(df_clean) - len(df_result)}개")
df_result.head(10)

저장 완료: 1000개 채널 → YT_ChannelData_2026-03-28_clean.csv
channel_id 없어서 제외: 0개


,channel_name,channel_id,subscribers,avg_daily_views
0,다이어트 과학자 최겸 Gyum Choi,UChXDq9Izwq-pTzeUWzwXL3A,603000,79982
1,한의사트레이너,UC_KjTyC5e0DFNxn6GIgBxcQ,144000,35569
2,블락스blocks,UC5qF47U0ErUHBgK4Rgy5ymA,10100,294
3,김지만TV,UCCFD5pLnLB2oEb4tPm1YemA,141000,995
4,김소영건강체조티비-Health gymnastics,UCSi2gkWNFwoMpb5Nu19EnpA,53000,16501
5,통증교정 수연쌤,UCafAiwzaq1wXHIJOCm4zSoQ,12400,1055
6,우수한 영양학,UCxkwgIYCm3vLlYITOzdByZg,62800,7539
7,빵느,UCRrZ5RYIalHLiHq5ftzxM6A,711000,66720
8,홍삼아 댄스&에어로빅,UCNcl4-NKaNrX0CBK3Gmd5Ag,21400,3136
9,약초1번지,UCvf59uLGe-VyEcoNsp15K0w,102000,532
